## LangChain实现RAG

**Document loaders文档加载模块**

LangChain封装了一系列类型的文档加载模块，例如PDF、CSV、HTML、JSON、Markdown、File Directory等。下面以PDF文件夹在为例看一下用法，其它类型的文档加载的用法都类似

LangChain加载PDF文件使用的是pypdf，先安装

pip install pypdf

加载PDF文件

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
load_dotenv()
import os

loader = PyPDFLoader("llama2.pdf")
# pages = loader.load_and_split()
pages = loader.load()

print(pages[:5])
print(len(pages))

# print(f"第0页：\n{pages[0]}") ## 也可通过 pages[0].page_content只获取本页内容
# Document 包含：page_content (文本) 和 metadata (元数据)
# page_content：真正会送进大模型参与推理的文本内容
# metadata：描述这段文本,从哪来、是什么、怎么用的辅助信息（默认不会被模型直接理解）

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'llama2.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}, page_content='Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev\nPunit Singh Koura Marie-Anne Lachaux Thibaut La

**加载在线的PDF文件**

需要安装的模块

```
pip install unstructured -i https://pypi.tuna.tsinghua.edu.cn/simple 
pip install pdf2image
pip install opencv-python
pip install unstructured-inference
pip install pikepdf
```

In [ ]:
from langchain_community.document_loaders import OnlinePDFLoader

loader = OnlinePDFLoader("https://arxiv.org/pdf/2302.03803.pdf")
data = loader.load()
# print(data)

可能会遇到的问题：ImportError: DLL load failed while importing onnx_cpp2py_export :动态链接库（DLL）初始化历程失败

需要将高版本的onnx卸载，在pip install onnx==1.16.1

**常用 Loaders**：
- `TextLoader` - 文本文件
- `PyPDFLoader` - PDF 文件
- `WebBaseLoader` - 网页
- `CSVLoader` - CSV 文件

In [7]:
from langchain_community.document_loaders import TextLoader

# 加载文本文件
loader = TextLoader("document.txt", encoding="utf-8")
documents = loader.load()
print(documents)

[Document(metadata={'source': 'document.txt'}, page_content='hello langchain')]


**Text Splitting 文档切分模块**

LangChain提供了许多不同类型的文本切分器

官网地址：https://python.langchain.com/api_reference/text_splitters/index.html

这里以Recursive为例展示用法。RecursiveCharacterTextSplitter是LangChain对这种文档切分方式的封装，里面的几个重点参数
- chunk_size：每个切块的token数量
- chunk_overlap：相邻两个切块之间重复的token数量

In [9]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./llama2.pdf")
pages = loader.load_and_split()
print(f"第1页：\n{pages[0].page_content}")
print("="*50)
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
)

paragraphs = text_splitter.create_documents([pages[0].page_content])
for para in paragraphs:
    print(para.page_content)
    print('-------')

第1页：
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic
Sergey E

In [ ]:
这里提供了一个可视化展示文本如何分割的工具，https://chunkviz.up.railway.app/

**Text embedding models 文本向量化模型封装**

LangChain对一些文本向量化模型的接口做了封装，例如OpenAI, Cohere, Hugging Face等。 向量化模型的封装提供了两种接口，一种针对文档的向量化embed_documents，一种针对句子的向量化embed_query

文档的向量化embed_documents，接收的参数是字符串数组

In [10]:
from langchain_openai import OpenAIEmbeddings

# embeddings_model = OpenAIEmbeddings()  ## OpenAI文本向量化模型接口的封装

from langchain_community.embeddings.dashscope import DashScopeEmbeddings
embeddings_model = DashScopeEmbeddings(
    model="text-embedding-v3",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

embeddings = embeddings_model.embed_documents(
    [
        "Hi there!",
        "Oh, hello!",
        "What's your name?",
        "My friends call me World",
        "Hello World!"
    ]
)

print(len(embeddings), len(embeddings[0]))
print(embeddings[0][:5])
##运行结果 (5, 1024)

5 1024
[-0.07202166318893433, 0.026690827682614326, -0.09668558090925217, -0.030630990862846375, -0.041485387831926346]


句子的向量化embed_query，接收的参数是字符串

In [11]:
embedded_query = embeddings_model.embed_query("What was the name mentioned in the conversation?")
print(embedded_query[:5])

[-0.05957995727658272, 0.014588589780032635, -0.0708080306649208, -0.0267457477748394, -0.049972839653491974]


**Vector stores 向量存储（数据库）**

将文本向量化之后，下一步就是进行向量的存储。

这部分包含两块：一是向量的存储。二是向量的查询。

官方提供了三种开源、免费的可用于本地机器的向量数据库示例（chroma、FAISS、 Lance）

安装模块：pip install langchain-chroma

创建向量数据库，灌入数据

In [12]:
from langchain_openai import OpenAIEmbeddings
# from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma
# db = Chroma.from_documents(paragraphs, embeddings_model) ## 一行代码搞定    

db = Chroma(collection_name="mydb", embedding_function=embeddings_model)

batch_size = 10
for i in range(0, len(paragraphs), batch_size):
    batch = paragraphs[i:i+batch_size]
    db.add_documents(batch)

查询similarity_search

In [13]:
query = "llama2有多少参数？"
docs = db.similarity_search(query) ## 一行代码搞定
for doc in docs:
    print(f"{doc.page_content}\n-------\n")

large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters.
Our fine-tuned LLMs, calledLlama 2-Chat, are optimized for dialogue use cases. Our
-------

Sergey Edunov Thomas Scialom∗
GenAI, Meta
Abstract
In this work, we develop and release Llama 2, a collection of pretrained and fine-tuned
-------

Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
-------

source models. We provide a detailed description of our approach to fine-tuning and safety
improvements ofLlama 2-Chat in order to enable the community to build on our work and
-------



**Retrievers 检索器**

检索器是在给定非结构化查询的情况下返回相关文本的接口。它比Vector stores更通用。检索器不需要能够存储文档，只需要返回（或检索）文档即可。Vector stores可以用作检索器的主干，但也有其他类型的检索器。检索器接受字符串查询作为输入，并返回文档列表作为输出

https://python.langchain.com/api_reference/langchain/retrievers.html#module-langchain.retrievers

<table><thead><tr><th>名称</th><th>索引类型</th><th>使用LLM</th><th>何时使用</th><th>描述</th></tr></thead><tbody><tr><td>Vectorstore</td><td>Vectorstore</td><td>否</td><td>刚开始接触，想快速简单的入门</td><td>这是最简单的方法，也是最容易入门的方法。它涉及为每个文本片段创建向量。</td></tr><tr><td>ParentDocument</td><td>Vectorstore + 文档存储</td><td>否</td><td>如果您的页面有许多较小的独立信息片段，最好单独索引，但最好一起检索。</td><td>这涉及为每个文档索引多个片段。然后找到在嵌入空间中最相似的片段，但检索整个父文档并返回（而不是单个片段）。</td></tr><tr><td>Multi Vector</td><td>Vectorstore + 文档存储</td><td>有时在索引过程中</td><td>如果您能够从文档中提取比文本本身更相关的信息。</td><td>这涉及为每个文档创建多个向量。每个向量可以以多种方式创建 - 例如，文本摘要和假设性问题。</td></tr><tr><td>Self Query</td><td>Vectorstore</td><td>是</td><td>如果用户提出的问题更适合根据元数据而不是文本相似性来获取文档回答。</td><td>这使用LLM将用户输入转换为两件事：（1）语义查找的字符串，（2）与之配套的元数据过滤器。这很有用，因为通常问题是关于文档的元数据（而不是内容本身）。</td></tr><tr><td>Contextual Compression</td><td>任意</td><td>有时</td><td>如果您发现检索到的文档包含太多不相关信息，并且干扰了LLM。</td><td>这在另一个检索器之上放置了后处理步骤，并仅从检索到的文档中提取最相关的信息。这可以使用嵌入或LLM完成。</td></tr><tr><td>Time-Weighted Vectorstore</td><td>Vectorstore</td><td>否</td><td>如果您的文档有时间戳，并且您想检索最近的文档。</td><td>这根据语义相似性（与常规向量检索相同）和时间权重（查看索引文档的时间戳）检索文档。</td></tr><tr><td>Multi-Query Retriever</td><td>任意</td><td>是</td><td>如果用户提出的问题复杂，并且需要多个独立信息片段来回应。</td><td>这使用LLM从原始查询生成多个查询。当原始查询需要有关多个主题的信息片段才能正确回答时，这很有用。通过生成多个查询，我们可以为每个查询获取文档。</td></tr><tr><td>Ensemble</td><td>任意</td><td>否</td><td>如果您有多个检索方法并希望尝试将它们组合起来。</td><td>这从多个检索器中获取文档，然后将它们组合在一起。</td></tr><tr><td>Long-Context Reorder</td><td>任意</td><td>否</td><td>如果您使用长上下文模型，并且注意到它没有关注检索到的文档中的中间信息。</td><td>这从基础检索器中获取文档，然后重新排序文档，使最相似的文档靠近开头和结尾。这很有用，因为长上下文模型有时不关注上下文窗口中间的信息。</td></tr></tbody></table>

以Vectorstore类型的检索为例简单使用

In [16]:
# as_retriever：生成检索器实例
retriever = db.as_retriever()
docs = retriever.invoke("llama2有多少参数？")
for doc in docs:
    print(f"{doc.page_content}\n-------\n")

Llama 2
-------

Llama 2
-------

but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
-------

Llama 2-Chat
-------



指定一个相似度阈值为0.5，只有相似度超过这个值才会召回

In [17]:
retriever = db.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.5}
)
docs = retriever.invoke("llama2有多少参数？")
print(docs)

[Document(metadata={'start_index': 1469}, page_content='Llama 2'), Document(metadata={'start_index': 1004}, page_content='Llama 2'), Document(metadata={'start_index': 1127}, page_content='but are not releasing.§\n2. Llama 2-Chat, a fine-tuned version ofLlama 2that is optimized for dialogue use cases. We release\nvariants of this model with 7B, 13B, and 70B parameters as well.'), Document(metadata={'start_index': 1967}, page_content='Llama 2-Chat')]


指定检索几个文本片段：topK

In [20]:
retriever = db.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("llama2有多少参数？")
print(docs)

[Document(metadata={'start_index': 1469}, page_content='Llama 2'), Document(metadata={'start_index': 1004}, page_content='Llama 2'), Document(metadata={'start_index': 1127}, page_content='but are not releasing.§\n2. Llama 2-Chat, a fine-tuned version ofLlama 2that is optimized for dialogue use cases. We release\nvariants of this model with 7B, 13B, and 70B parameters as well.')]


**RAG问答**

In [15]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# 将 vectorstore 封装为工具
@tool
def search_kb(query: str) -> str:
    """搜索知识库"""
    docs = db.similarity_search(query, k=3)
    return "\n\n".join([doc.page_content for doc in docs])

model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)

# 创建 RAG Agent
agent = create_agent(
    model=model,
    tools=[search_kb],
    system_prompt="""你是助手。使用search_kb工具检索信息，然后回答问题。"""
)

# 问答
response = agent.invoke({
    "messages": [{"role": "user", "content": "llama2有多少参数？"}]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

llama2有多少参数？
================================== Ai Message ==================================

我来帮您查询llama2的参数数量信息。
Tool Calls:
  search_kb (call_00_fAiP5GoMBFukjONZdu6oQ9hN)
 Call ID: call_00_fAiP5GoMBFukjONZdu6oQ9hN
  Args:
    query: llama2 参数数量 参数量
================================= Tool Message =================================
Name: search_kb

large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters.
Our fine-tuned LLMs, calledLlama 2-Chat, are optimized for dialogue use cases. Our

Sergey Edunov Thomas Scialom∗
GenAI, Meta
Abstract
In this work, we develop and release Llama 2, a collection of pretrained and fine-tuned

source models. We provide a detailed description of our approach to fine-tuning and safety
improvements ofLlama 2-Chat in order to enable the community to build on our work and
================================== Ai Message =================================